In [0]:
-- Ensure Gold Schema Exists
CREATE SCHEMA IF NOT EXISTS dataco.gold;

-- =========================================================================
-- 1. DIMENSION: DimCustomer
-- =========================================================================
CREATE TABLE IF NOT EXISTS dataco.gold.DimCustomer (
    CustomerID INT,
    FirstName STRING,
    LastName STRING,
    Email STRING,
    Segment STRING,
    Street STRING,
    City STRING,
    State STRING,
    Country STRING
);

MERGE INTO dataco.gold.DimCustomer AS target
USING (
    SELECT DISTINCT
        Customer_Id          AS CustomerID,
        Customer_Fname       AS FirstName,
        Customer_Lname       AS LastName,
        Customer_Email       AS Email,
        Customer_Segment     AS Segment,
        Customer_Street      AS Street,
        Customer_City        AS City,
        Customer_State       AS State,
        Customer_Country     AS Country
    FROM dataco.silver.enriched_supply_chain
    WHERE Customer_Id IS NOT NULL
) AS source
ON target.CustomerID = source.CustomerID
WHEN MATCHED THEN UPDATE SET
    target.FirstName = source.FirstName,
    target.LastName  = source.LastName,
    target.Email     = source.Email,
    target.Segment   = source.Segment,
    target.Street    = source.Street,
    target.City      = source.City,
    target.State     = source.State,
    target.Country   = source.Country
WHEN NOT MATCHED THEN INSERT (
    CustomerID, FirstName, LastName, Email, Segment, Street, City, State, Country
) VALUES (
    source.CustomerID, source.FirstName, source.LastName, source.Email, source.Segment, source.Street, source.City, source.State, source.Country
);


In [0]:
-- =========================================================================
-- 2. DIMENSION: DimProduct
-- =========================================================================
CREATE TABLE IF NOT EXISTS dataco.gold.DimProduct (
    ProductID INT,
    ProductName STRING,
    CategoryID INT,
    CategoryName STRING,
    ProductPrice DOUBLE,
    ProductStatus INT
);

MERGE INTO dataco.gold.DimProduct AS target
USING (
    SELECT DISTINCT
        Product_Card_Id      AS ProductID,
        Product_Name         AS ProductName,
        Product_Category_Id  AS CategoryID,
        Category_Name        AS CategoryName,   
        Product_Price        AS ProductPrice,
        Product_Status       AS ProductStatus
    FROM dataco.silver.enriched_supply_chain
    WHERE Product_Card_Id IS NOT NULL
) AS source
ON target.ProductID = source.ProductID
WHEN MATCHED THEN UPDATE SET
    target.ProductName   = source.ProductName,
    target.CategoryID    = source.CategoryID,
    target.CategoryName  = source.CategoryName,
    target.ProductPrice  = source.ProductPrice,
    target.ProductStatus = source.ProductStatus
WHEN NOT MATCHED THEN INSERT (
    ProductID, ProductName, CategoryID, CategoryName, ProductPrice, ProductStatus
) VALUES (
    source.ProductID, source.ProductName, source.CategoryID, source.CategoryName, source.ProductPrice, source.ProductStatus
);

In [0]:
-- =========================================================================
-- 3. DIMENSION: DimSupplier (Department)
-- =========================================================================
CREATE OR REPLACE TABLE dataco.gold.DimSupplier AS
SELECT DISTINCT
    Department_Id        AS DepartmentID,
    Department_Name      AS DepartmentName,
    Latitude             AS Latitude,
    Longitude            AS Longitude
FROM dataco.silver.enriched_supply_chain
WHERE Department_Id IS NOT NULL;

In [0]:
-- =========================================================================
-- 4. DIMENSION: DimDate
-- =========================================================================
CREATE TABLE IF NOT EXISTS dataco.gold.DimDate (
    DateKey INT,
    FullDate DATE,
    Year INT,
    Month INT,
    MonthName STRING,
    Quarter INT,
    DayOfMonth INT,
    DayOfWeekName STRING
);

MERGE INTO dataco.gold.DimDate AS target
USING (
    SELECT DISTINCT
        CAST(date_format(order_date_DateOrders, 'yyyyMMdd') AS INT) AS DateKey,
        CAST(order_date_DateOrders AS DATE)                         AS FullDate,
        YEAR(order_date_DateOrders)                                 AS Year,
        MONTH(order_date_DateOrders)                                AS Month,
        date_format(order_date_DateOrders, 'MMMM')                  AS MonthName,
        QUARTER(order_date_DateOrders)                              AS Quarter,
        DAY(order_date_DateOrders)                                  AS DayOfMonth,
        date_format(order_date_DateOrders, 'EEEE')                  AS DayOfWeekName
    FROM dataco.silver.enriched_supply_chain
    WHERE order_date_DateOrders IS NOT NULL
) AS source
ON target.DateKey = source.DateKey
WHEN NOT MATCHED THEN INSERT (
    DateKey, FullDate, Year, Month, MonthName, Quarter, DayOfMonth, DayOfWeekName
) VALUES (
    source.DateKey, source.FullDate, source.Year, source.Month, source.MonthName, source.Quarter, source.DayOfMonth, source.DayOfWeekName
);

In [0]:
-- =========================================================================
-- 5. DIMENSION: DimShippingMode
-- =========================================================================
CREATE OR REPLACE TABLE dataco.gold.DimShippingMode AS
SELECT DISTINCT
    DENSE_RANK() OVER (ORDER BY Shipping_Mode) AS ShippingModeID,
    Shipping_Mode                              AS ShippingMode,
    Days_for_shipping_real                     AS DaysForShippingReal,
    Days_for_shipment_scheduled                AS DaysForShipmentScheduled,
    Delivery_Status                            AS DeliveryStatus,
    Late_delivery_risk                         AS LateDeliveryRisk
FROM dataco.silver.enriched_supply_chain
WHERE Shipping_Mode IS NOT NULL;

In [0]:
-- =========================================================================
-- 6. DIMENSION: DimRegion
-- =========================================================================
CREATE OR REPLACE TABLE dataco.gold.DimRegion AS
SELECT DISTINCT
    DENSE_RANK() OVER (ORDER BY Market, Order_Region, Order_Country) AS RegionID,
    Market        AS Market,
    Order_Region  AS OrderRegion,
    Order_Country AS OrderCountry,
    Order_State   AS OrderState,
    Order_City    AS OrderCity
FROM dataco.silver.enriched_supply_chain
WHERE Market IS NOT NULL;

In [0]:
-- =========================================================================
-- 7. FACT TABLE: FactOrders
-- =========================================================================
CREATE TABLE IF NOT EXISTS dataco.gold.FactOrders (
    OrderItemID INT,
    OrderID INT,
    CustomerID INT,
    ProductID INT,
    DepartmentID INT,
    OrderDateKey INT,
    Market STRING,
    OrderStatus STRING,
    Quantity INT,
    UnitPrice DOUBLE,
    DiscountAmount DOUBLE,
    DiscountRate DOUBLE,
    GrossSales DOUBLE,
    NetSales DOUBLE,
    OrderProfit DOUBLE,
    ProfitRatio DOUBLE,
    OrderRegion STRING
);

MERGE INTO dataco.gold.FactOrders AS target
USING (
    SELECT 
        Order_Item_Id                          AS OrderItemID,
        Order_Id                               AS OrderID,
        Order_Customer_Id                      AS CustomerID,
        Product_Card_Id                        AS ProductID,
        Department_Id                          AS DepartmentID,
        CAST(date_format(order_date_DateOrders, 'yyyyMMdd') AS INT) AS OrderDateKey,
        Market                                 AS Market,
        Order_Status                           AS OrderStatus,
        Order_Item_Quantity                    AS Quantity,
        Order_Item_Product_Price               AS UnitPrice,
        Order_Item_Discount                    AS DiscountAmount,
        Order_Item_Discount_Rate               AS DiscountRate,
        Sales                                  AS GrossSales,
        Order_Item_Total                       AS NetSales,
        Order_Profit_Per_Order                 AS OrderProfit,
        Order_Item_Profit_Ratio                AS ProfitRatio,
        Order_Region                           AS OrderRegion
    FROM dataco.silver.enriched_supply_chain
    WHERE Order_Item_Id IS NOT NULL
) AS source
ON target.OrderItemID = source.OrderItemID
WHEN MATCHED THEN UPDATE SET
    target.OrderStatus    = source.OrderStatus,
    target.Quantity       = source.Quantity,
    target.UnitPrice      = source.UnitPrice,
    target.DiscountAmount = source.DiscountAmount,
    target.DiscountRate   = source.DiscountRate,
    target.GrossSales     = source.GrossSales,
    target.NetSales       = source.NetSales,
    target.OrderProfit    = source.OrderProfit,
    target.ProfitRatio    = source.ProfitRatio
WHEN NOT MATCHED THEN INSERT (
    OrderItemID, OrderID, CustomerID, ProductID, DepartmentID, OrderDateKey, Market, OrderStatus, Quantity, UnitPrice, DiscountAmount, DiscountRate, GrossSales, NetSales, OrderProfit, ProfitRatio, OrderRegion
) VALUES (
    source.OrderItemID, source.OrderID, source.CustomerID, source.ProductID, source.DepartmentID, source.OrderDateKey, source.Market, source.OrderStatus, source.Quantity, source.UnitPrice, source.DiscountAmount, source.DiscountRate, source.GrossSales, source.NetSales, source.OrderProfit, source.ProfitRatio, source.OrderRegion
);

In [0]:
-- =========================================================================
-- 8. FACT TABLE: FactShipments
-- =========================================================================
CREATE TABLE IF NOT EXISTS dataco.gold.FactShipments (
    OrderID INT,
    OrderItemID INT,
    CustomerID INT,
    ShipDateKey INT,
    ShippingMode STRING,
    DeliveryStatus STRING,
    LateDeliveryRisk INT,
    ScheduledDays INT,
    ActualDays INT,
    DeliveryDelayDays INT
);

MERGE INTO dataco.gold.FactShipments AS target
USING (
    SELECT 
        Order_Id                               AS OrderID,
        Order_Item_Id                          AS OrderItemID,
        Order_Customer_Id                      AS CustomerID,
        CAST(date_format(shipping_date_DateOrders, 'yyyyMMdd') AS INT) AS ShipDateKey,
        Shipping_Mode                          AS ShippingMode,
        Delivery_Status                        AS DeliveryStatus,
        Late_delivery_risk                     AS LateDeliveryRisk,
        Days_for_shipment_scheduled            AS ScheduledDays,
        Days_for_shipping_real                 AS ActualDays,
        (Days_for_shipping_real - Days_for_shipment_scheduled) AS DeliveryDelayDays
    FROM dataco.silver.enriched_supply_chain
    WHERE Order_Item_Id IS NOT NULL
) AS source
ON target.OrderItemID = source.OrderItemID
WHEN MATCHED THEN UPDATE SET
    target.ShipDateKey       = source.ShipDateKey,
    target.ShippingMode      = source.ShippingMode,
    target.DeliveryStatus    = source.DeliveryStatus,
    target.LateDeliveryRisk  = source.LateDeliveryRisk,
    target.ScheduledDays     = source.ScheduledDays,
    target.ActualDays        = source.ActualDays,
    target.DeliveryDelayDays = source.DeliveryDelayDays
WHEN NOT MATCHED THEN INSERT (
    OrderID, OrderItemID, CustomerID, ShipDateKey, ShippingMode, DeliveryStatus, LateDeliveryRisk, ScheduledDays, ActualDays, DeliveryDelayDays
) VALUES (
    source.OrderID, source.OrderItemID, source.CustomerID, source.ShipDateKey, source.ShippingMode, source.DeliveryStatus, source.LateDeliveryRisk, source.ScheduledDays, source.ActualDays, source.DeliveryDelayDays
);

In [0]:
-- =========================================================================
-- 9. AGGREGATED FACT TABLE: FactSales
-- Note: Recomputed full aggregation per load (Overwrites target)
-- =========================================================================
CREATE OR REPLACE TABLE dataco.gold.FactSales AS
SELECT 
    Order_Customer_Id                      AS CustomerID,
    Product_Card_Id                        AS ProductID,
    CAST(date_format(order_date_DateOrders, 'yyyyMMdd') AS INT) AS OrderDateKey,
    Market                                 AS Market,
    COUNT(DISTINCT Order_Id)               AS TotalOrders,
    SUM(Order_Item_Quantity)               AS TotalUnitsSold,
    SUM(Sales)                             AS GrossSalesAmount,
    SUM(Order_Item_Discount)               AS TotalDiscountAmount,
    SUM(Order_Item_Total)                  AS NetSalesAmount,
    SUM(Benefit_per_order)                 AS TotalProfit,
    AVG(Order_Item_Profit_Ratio)           AS AvgProfitMargin
FROM dataco.silver.enriched_supply_chain
WHERE Order_Item_Id IS NOT NULL
GROUP BY 
    Order_Customer_Id,
    Product_Card_Id,
    CAST(date_format(order_date_DateOrders, 'yyyyMMdd') AS INT),
    Market;